# Interactive Graphics: Mouse Clicks and a Target Game

Last time, we used functions to draw shapes on a canvas. Today, we will make our programs **respond to the user**.

By the end of class, we will be able to:

- wait for a mouse click,
- retrieve the coordinates of a click,
- use conditionals to decide what the user clicked,
- divide a program into small functions, and
- build a three-target game that records a score.

## The new mouse functions

The graphics library records the most recent place where the user clicked.

- `wait_for_mouse_click()` pauses the program until the canvas is clicked.
- `get_mouse_click_x()` returns the x-coordinate of the most recent click.
- `get_mouse_click_y()` returns the y-coordinate of the most recent click.

The `get` functions should normally be called **after** `wait_for_mouse_click()`.

In [ ]:
from cs1.graphics import *


def main():
    open_canvas(500, 300)

    set_color("black")
    draw_string("Click anywhere on the canvas", 250, 40, 20)

    wait_for_mouse_click()

    click_x = get_mouse_click_x()
    click_y = get_mouse_click_y()

    set_color("blue")
    draw_filled_circle(click_x, click_y, 6)

    set_color("black")
    message = "You clicked (" + str(click_x) + ", " + str(click_y) + ")"
    draw_string(message, 250, 280, 16)


main()

## Was a rectangular button clicked?

Suppose a button has its upper-left corner at $(x,y)$ and has a given width and height. A click at $(c_x,c_y)$ is inside the button when

$$
x \leq c_x \leq x + \text{width}
\qquad\text{and}\qquad
y \leq c_y \leq y + \text{height}.
$$

In Python, we can write this test as a function:

```python
def is_inside_rectangle(click_x, click_y, x, y, width, height):
    if (x <= click_x <= x + width and
            y <= click_y <= y + height):
        return True
    else:
        return False
```

`return` sends an answer back to the part of the program that called the function.

In [ ]:
from cs1.graphics import *


def is_inside_rectangle(click_x, click_y, x, y, width, height):
    if (x <= click_x <= x + width and
            y <= click_y <= y + height):
        return True
    else:
        return False


def main():
    open_canvas(500, 250)

    set_color("royalblue")
    draw_filled_rect(175, 80, 150, 60)

    set_color("white")
    draw_string("Click me!", 250, 115, 20)

    wait_for_mouse_click()
    click_x = get_mouse_click_x()
    click_y = get_mouse_click_y()

    if is_inside_rectangle(click_x, click_y, 175, 80, 150, 60):
        set_color("green")
        draw_string("You clicked the button!", 250, 190, 18)
    else:
        set_color("red")
        draw_string("You missed the button.", 250, 190, 18)


main()

## Exercise 1: Choose a color

Draw two rectangular buttons:

- a red button on the left,
- a blue button on the right.

Wait for one click. If the user clicks the red button, set the background to red. If the user clicks the blue button, set the background to blue. Otherwise, display `You missed both buttons.`

Use `is_inside_rectangle()` from the previous example.

In [ ]:
from cs1.graphics import *


def is_inside_rectangle(click_x, click_y, x, y, width, height):
    # Complete this function or reuse the earlier version.
    pass


def main():
    open_canvas(500, 250)

    # 1. Draw the two buttons.
    # 2. Wait for a click and get its coordinates.
    # 3. Use if, elif, and else to respond to the click.
    pass


main()

## Was a circular target clicked?

For a target centered at $(t_x,t_y)$, compare the click-to-center distance with the radius $r$:

$$
(c_x-t_x)^2+(c_y-t_y)^2 \leq r^2.
$$

We do not need to calculate the square root. If this inequality is true, the click is inside or on the target.

In [ ]:
def is_target_hit(click_x, click_y, target_x, target_y, radius):
    distance_squared = ((click_x - target_x) ** 2
                        + (click_y - target_y) ** 2)

    if distance_squared <= radius ** 2:
        return True
    else:
        return False

## Exercise 2: Test one target

Write a program that:

1. Opens a $500\times300$ canvas.
2. Draws a filled target centered at $(250,150)$ with radius $40$.
3. Waits for one click.
4. Draws a small blue dot where the user clicked.
5. Displays `Hit!` or `Miss!` using `is_target_hit()`.

If you finish early, draw two rings around the target to make it look like a bullseye.

In [ ]:
from cs1.graphics import *


def is_target_hit(click_x, click_y, target_x, target_y, radius):
    # Complete this function or reuse the earlier version.
    pass


def main():
    # Complete the program.
    pass


main()

# Instructor Demonstration: Three-Target Game

The following game displays **three different targets**, one at a time: a circle, a rectangle, and a star. The closer the click is to the center of any shape, the more points it earns:

$$
\text{points}
=
\max\left(0,\operatorname{round}(20-\text{distance})\right).
$$

An exact center hit earns 20 points. A click 10 pixels from the center earns about 10 points, and a click 20 or more pixels away earns 0 points. The scoring rule uses the center point, so it works the same way for every shape. After the third attempt, the player sees a final score out of 60.

The **Reset Game** button can be clicked during any round or after the final score. Resetting clears the score and starts again with three new targets.

Most of the program uses the ideas from today's exercises. The `while` loop near the bottom allows the reset button to restart the game repeatedly; loops will be studied separately.

In [ ]:
from cs1.graphics import *
from random import randint


CANVAS_WIDTH = 500
CANVAS_HEIGHT = 350
TARGET_RADIUS = 20

RESET_X = 375
RESET_Y = 10
RESET_WIDTH = 110
RESET_HEIGHT = 35


def is_inside_rectangle(click_x, click_y, x, y, width, height):
    if (x <= click_x <= x + width and
            y <= click_y <= y + height):
        return True
    else:
        return False


def is_target_hit(click_x, click_y, target_x, target_y, radius):
    distance_squared = ((click_x - target_x) ** 2
                        + (click_y - target_y) ** 2)

    if distance_squared <= radius ** 2:
        return True
    else:
        return False


def points_for_target(click_x, click_y, target_x, target_y):
    distance_squared = ((click_x - target_x) ** 2
                        + (click_y - target_y) ** 2)
    distance = distance_squared ** 0.5

    if distance <= TARGET_RADIUS:
        return round(20 - distance)
    else:
        return 0


def draw_reset_button():
    set_color("darkgray")
    draw_filled_rect(RESET_X, RESET_Y, RESET_WIDTH, RESET_HEIGHT)

    set_color("white")
    draw_string("Reset Game", RESET_X + RESET_WIDTH / 2,
                RESET_Y + 23, 15)


def draw_score(score, round_number):
    set_color("black")
    draw_string("Score: " + str(score), 60, 30, 18)
    draw_string("Target " + str(round_number) + " of 3", 245, 30, 18)


def draw_circle_target(target_x, target_y):
    set_color("red")
    draw_filled_circle(target_x, target_y, TARGET_RADIUS)


def draw_rectangle_target(target_x, target_y):
    set_color("green")
    draw_filled_rect(target_x - 20, target_y - 20, 40, 40)


def draw_star_target(target_x, target_y):
    set_color("purple")
    draw_filled_polygon(
        target_x, target_y - 22,
        target_x + 5, target_y - 7,
        target_x + 21, target_y - 7,
        target_x + 8, target_y + 3,
        target_x + 13, target_y + 19,
        target_x, target_y + 9,
        target_x - 13, target_y + 19,
        target_x - 8, target_y + 3,
        target_x - 21, target_y - 7,
        target_x - 5, target_y - 7
    )


def draw_target(target_x, target_y, target_shape):
    if target_shape == "circle":
        draw_circle_target(target_x, target_y)
    elif target_shape == "rectangle":
        draw_rectangle_target(target_x, target_y)
    else:
        draw_star_target(target_x, target_y)


def play_round(score, round_number, target_shape):
    clear_canvas()
    draw_score(score, round_number)
    draw_reset_button()

    target_x = randint(50, 450)
    target_y = randint(90, 300)
    draw_target(target_x, target_y, target_shape)

    wait_for_mouse_click()
    click_x = get_mouse_click_x()
    click_y = get_mouse_click_y()

    if is_inside_rectangle(click_x, click_y,
                           RESET_X, RESET_Y,
                           RESET_WIDTH, RESET_HEIGHT):
        return -1

    set_color("blue")
    draw_filled_circle(click_x, click_y, 5)

    points = points_for_target(click_x, click_y, target_x, target_y)
    return score + points


def play_three_targets():
    score = 0

    score = play_round(score, 1, "circle")
    if score == -1:
        return -1

    score = play_round(score, 2, "rectangle")
    if score == -1:
        return -1

    score = play_round(score, 3, "star")
    if score == -1:
        return -1

    clear_canvas()
    draw_reset_button()

    set_color("purple")
    draw_string("Final score: " + str(score) + " out of 60",
                250, 155, 28)
    set_color("black")
    draw_string("Click Reset Game to play again",
                250, 205, 17)

    wait_for_mouse_click()
    click_x = get_mouse_click_x()
    click_y = get_mouse_click_y()

    if is_inside_rectangle(click_x, click_y,
                           RESET_X, RESET_Y,
                           RESET_WIDTH, RESET_HEIGHT):
        return -1
    else:
        return score


def main():
    open_canvas(CANVAS_WIDTH, CANVAS_HEIGHT)
    set_background_color("lightyellow")

    reset_requested = True

    # This loop lets the Reset Game button restart the game repeatedly.
    while reset_requested:
        result = play_three_targets()

        if result == -1:
            reset_requested = True
        else:
            reset_requested = False


main()

## Reading the game program

Before changing the code, identify the responsibility of each function:

- Which functions only draw something?
- Which functions answer a `True` or `False` question?
- Which function returns the number of points earned?
- How does `draw_target()` decide which shape to draw?
- Which function plays exactly one round?
- Which function records the score across all three rounds?
- Why does `play_round()` return `-1` when Reset Game is clicked?

Then try one modification:

1. Make the target smaller.
2. Change the maximum distance or maximum number of points.
3. Change the background and target colors.
4. Add a fourth target shape.

# Final Exercise: Create Your Own Clicking Game

Create your own interactive game using mouse clicks. You may use the **Three-Target Game** as your starting point and reuse or modify any of its code.

Your game could ask the player to:

- click the centers of different shapes to earn points,
- click several buttons in the correct order,
- follow instructions about which object to click,
- avoid clicking certain objects,
- choose between different objects,
- or complete another clicking challenge of your own design.

## Requirements

Your game must:

1. Display clear instructions on the canvas.
2. Include at least **five clickable targets or choices**.
3. Use at least **three different shapes, colors, or sizes**.
4. Use mouse-click functions to receive the player's choices.
5. Use conditionals to decide what happens after each click.
6. Use functions to organize the program.
7. Keep track of a score, progress, or number of correct choices.
8. Display a final result when the game ends.
9. Include a button that allows the player to reset or replay the game.

You are welcome to copy functions from the instructor demonstration. However, you must change the game enough to make it your own.

Before writing code, decide:

- What is the goal of your game?
- What should the player click?
- What earns or loses points?
- How does the player know whether a choice was correct?
- When does the game end?
- What happens when the player resets the game?

Be creative, but keep the game simple enough that you can finish and test it during class.